# Notebook C — Hétérogénéité cellulaire et extension TMG

**Notebook C — Stage de lycée · Laboratoire de microbiologie quantitative (version stagiaire)**

Ce notebook prolonge le **Notebook B** au niveau du système : B a mesuré comment
l'IPTG active le gène *lac* dans une population (moyenne de la plaque). Ici, on
regarde ce que l'IPTG ne montre pas — ce qui se passe quand l'inducteur (le TMG)
doit lui-même être transporté par une protéine codée par le gène qu'il active.

Structure :
1. **§C1 — Mécanisme** : la rétroaction LacY/TMG, et le rôle de l'architecture
   du promoteur (un seul opérateur comme lacUV5, ou une boucle ADN comme Plac).
2. **§C2 — Les 4 courbes dose-réponse, construites cellule par cellule** :
   lacUV5×IPTG, Plac×IPTG, lacUV5×TMG, Plac×TMG — en partant des trajectoires
   individuelles plutôt que d'une formule, comme au §B2.
3. **§C3 [FACULTATIF]** : si vous avez des images de microscopie, les analyser
   avec ImageJ et comparer à la prédiction du §C1.

---

> **Prérequis** : avoir complété le Notebook B.
> **Environnement** : [Google Colab](https://colab.research.google.com)
> **Durée** : pas de limite fixe — allez aussi loin que le temps le permet.
>  §C1 est la partie la plus importante si le temps manque.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size']  = 12
print("Bibliotheques chargees.")


## Section C1 — Mécanisme : rétroaction LacY / TMG

Le TMG est transporté dans la cellule par la protéine **LacY** — elle-même codée par
l'opéron *lac*. Cela crée une **boucle de rétroaction positive** :

```
Plus de LacY → plus de TMG intracellulaire → promoteur plus actif → plus de LacY
```

Cette boucle peut créer **deux états stables** :
- **OFF** : peu de LacY → peu de TMG intracellulaire → promoteur peu actif
- **ON**  : beaucoup de LacY → beaucoup de TMG → promoteur très actif

À des concentrations **intermédiaires** de TMG, les deux états coexistent.
Deux cellules identiques dans le même milieu peuvent finir dans des états différents
selon leur état initial — c'est la **bistabilité**.

> **Lien avec §B1 :** dans §B1, toutes les cellules avaient le même
> $p_{\text{on}}$ — leur niveau d'expression variait autour d'une même moyenne
> par hasard moléculaire. Ici, c'est différent : deux cellules identiques dans
> le même milieu peuvent finir dans des états très différents selon leur
> **état initial** — leur niveau d'expression au moment où le TMG arrive.
> Ce n'est plus du bruit autour d'une moyenne : ce sont deux états stables
> distincts, et chaque cellule converge vers l'un ou l'autre selon son point
> de départ.

L'IPTG n'a pas cette rétroaction : il entre par diffusion passive, donc son
niveau intracellulaire ne dépend pas de ce que la cellule a déjà produit.
Le modèle ci-dessous représente cette différence avec un paramètre `gain`
(`gain = 0` pour l'IPTG, `gain > 0` pour le TMG), et la coopérativité du
promoteur (un seul opérateur comme lacUV5, ou une boucle ADN comme Plac natif)
avec un paramètre `n_hill`.


In [ ]:
def simulate_cells(n_cells, n_t, K, gain, n_hill, conc_ext, alpha,
                    etat_initial=None, seed=None):
    # Simule l'evolution temporelle de n_cells cellules soumises a une
    # concentration externe d'inducteur conc_ext.
    #
    # Parametres :
    #   n_cells      — nombre de cellules simulees
    #   n_t          — nombre de pas de temps
    #   K            — demi-saturation de la reponse du promoteur (µM)
    #   gain         — force de la retro-action positive
    #                  (0 = pas de retro-action, comme l'IPTG ;
    #                   >0 = retro-action via LacY, comme le TMG)
    #   n_hill       — cooperativite de la reponse
    #                  (1 = un seul operateur, comme lacUV5 ;
    #                   2 = boucle ADN, comme Plac natif)
    #   conc_ext     — concentration d'inducteur dans le milieu (µM)
    #   alpha        — vitesse de relaxation vers l'etat cible
    #   etat_initial — valeur de depart commune (None = aleatoire pour
    #                  chaque cellule, entre 0 et 1)
    #   seed         — graine aleatoire (reproductibilite)
    #
    # Retourne : TABLEAU numpy de longueur n_cells — l'etat final
    #            (niveau d'expression relatif, entre 0 et 1) de chaque cellule.
    rng = np.random.default_rng(seed)
    if etat_initial is None:
        etat = rng.uniform(0, 1, n_cells)
    else:
        etat = np.full(n_cells, etat_initial, dtype=float)

    for t in range(n_t):
        conc_eff = conc_ext * (1 + gain * etat)
        p_on     = conc_eff**n_hill / (K**n_hill + conc_eff**n_hill)
        etat     = etat + alpha * (p_on - etat)
        etat     = np.clip(etat, 0, 1)

    return etat


In [ ]:
# Objectif : simuler le systeme de retro-action LacY/TMG pour deux cellules
# identiques au depart dans des etats differents, puis pour une population,
# afin de visualiser la bistabilite. Utiliser la fonction simulate_cells
# definie ci-dessus pour la partie population.
#
# Parametres :
#   K_bis    — demi-saturation (µM TMG intracellulaire)
#   gain_d   — facteur d'amplification du TMG par LacY (retro-action positive)
#   n_hill_d — cooperativite (2 pour cette demonstration, comme Plac natif —
#              voir §C1 (suite) pour ce qui se passe avec n_hill_d = 1)
#   alpha    — vitesse de relaxation de LacY vers sa valeur cible
#   n_t      — nombre de pas de temps
#   tmg_ext  — concentration de TMG extracellulaire (µM), choisie dans la
#              zone bistable (etroite : essayez d'abord tmg_ext = 5)
#
# Pour les DEUX TRAJECTOIRES (graphique de gauche), reproduire pas a pas la
# meme logique que simulate_cells, mais pour une seule cellule a la fois,
# en conservant tout l'historique (une LISTE traj par cellule, longueur n_t+1) :
#   conc_eff = tmg_ext * (1 + gain_d * lacy)
#   p_on     = conc_eff**n_hill_d / (K_bis**n_hill_d + conc_eff**n_hill_d)
#   lacy     = lacy + alpha * (p_on - lacy), puis borner entre 0 et 1
#
# Pour la POPULATION (graphique de droite), appeler simulate_cells avec
# n_cells=N_pop et etat_initial=None (depart aleatoire pour chaque cellule).
# Structure : un TABLEAU numpy lacy_finales de longueur N_pop.
#
# Graphique : 2 sous-graphiques cote a cote (plt.subplots(1, 2, ...)) —
#   Gauche : les deux trajectoires lacy(t).
#   Droite : histogramme des valeurs finales de lacy_finales.
#
# Afficher avec print() : que represente une distribution bimodale en termes
# de population cellulaire ? Essayer plusieurs valeurs de tmg_ext (2, 5, 6,
# 20 µM) — a quelle concentration la distribution devient-elle unimodale ?


### Coopérativité et rétro-action : faut-il les deux ?

Le modèle ci-dessus utilisait `n_hill = 2` (boucle ADN, comme Plac natif) avec
rétro-action (`gain = 20`, comme le TMG). Avant de construire les 4 courbes
complètes au §C2, testons les 3 autres combinaisons : est-ce que la
coopérativité seule (`n_hill = 2`, `gain = 0`) suffit à produire une
distribution bimodale ? Et la rétro-action seule (`n_hill = 1`, `gain > 0`),
sans coopérativité ?

> **Simplification** : les 4 panneaux ci-dessous utilisent le même K (celui
> du TMG) pour rester comparables entre eux. Au §C2, chaque inducteur aura
> son propre K, comme dans la réalité.


In [ ]:
# Objectif : comparer 4 combinaisons (n_hill, gain) pour determiner si la
# bimodalite vient de la cooperativite, de la retro-action, ou des deux.
#
# Parametres :
#   conc_test — concentration d'inducteur fixe pour cette comparaison (µM)
#   N_pop     — nombre de cellules simulees par panneau
#   configs   — LISTE de 4 tuples (n_hill, gain, titre) :
#     (1, 0,  "lacUV5 + IPTG  (n=1, sans retro-action)")
#     (2, 0,  "Plac + IPTG  (n=2, sans retro-action)")
#     (1, 20, "lacUV5 + TMG  (n=1, avec retro-action)")
#     (2, 20, "Plac + TMG  (n=2, avec retro-action)")
#
# Pour chaque element de configs, appeler simulate_cells(N_pop, n_t, K_bis,
# gain, n_hill, conc_test, alpha, seed=2) pour obtenir un TABLEAU numpy
# des etats finaux, puis tracer son histogramme.
#
# Graphique : grille 2x2 (plt.subplots(2, 2, ...)), un histogramme par
# combinaison, avec le titre correspondant.
#
# Afficher avec print() : quelle(s) combinaison(s) montre(nt) une distribution
# bimodale ? Est-ce la cooperativite (n), la retro-action (gain), ou les deux
# qui semblent necessaires ?


---
## Section C2 — Les 4 courbes dose-réponse, construites cellule par cellule

Au §B2, la courbe dose-réponse pour l'IPTG venait directement d'une formule
($p = [I]^n/(K^n+[I]^n)$), évaluée pour chaque concentration. Cette formule
fonctionne bien parce qu'il n'y a pas de rétro-action : chaque cellule converge
vers le même état, quel que soit son point de départ.

Avec le TMG, ce n'est plus vrai (cf. §C1) : il n'existe pas de formule fermée
aussi simple, parce que l'état final peut dépendre de l'état initial. La seule
façon de construire la courbe dose-réponse est de simuler beaucoup de cellules
individuelles, à chaque concentration, et de regarder la distribution de
leurs états finaux — d'où l'idée de construction **"bottom-up"** (de la cellule
individuelle vers la courbe de population), au lieu de la formule fermée
**"top-down"** utilisée en §B2.

On va construire les 4 courbes correspondant à votre plan expérimental réel :
deux souches (lacUV5-GFP, Plac-GFP) × deux inducteurs (IPTG, TMG).


In [ ]:
# ── Parametres des 4 combinaisons souche x inducteur (config) ──────────────────
concentrations_IPTG = [0, 1, 3, 10, 30, 100, 300]
concentrations_TMG  = [0, 1, 2, 4, 5, 5.5, 6, 8, 12, 20, 50]

combinaisons = {
    ('lacUV5-GFP', 'IPTG'): dict(n_hill=1, gain=0,  K=30, concs=concentrations_IPTG),
    ('Plac-GFP',   'IPTG'): dict(n_hill=2, gain=0,  K=30, concs=concentrations_IPTG),
    ('lacUV5-GFP', 'TMG'):  dict(n_hill=1, gain=20, K=50, concs=concentrations_TMG),
    ('Plac-GFP',   'TMG'):  dict(n_hill=2, gain=20, K=50, concs=concentrations_TMG),
}


In [ ]:
# Objectif : construire les 4 courbes dose-reponse "bottom-up" (a partir de
# cellules individuelles simulees), une par combinaison souche x inducteur
# du dictionnaire combinaisons defini ci-dessus.
#
# Parametres :
#   N_cells_curve — nombre de cellules simulees a chaque concentration
#   positions     — DICTIONNAIRE associant chaque combinaison a sa position
#                   dans une grille 2x2 :
#     ('lacUV5-GFP', 'IPTG'): (0, 0)   ('Plac-GFP', 'IPTG'): (0, 1)
#     ('lacUV5-GFP', 'TMG'):  (1, 0)   ('Plac-GFP', 'TMG'):  (1, 1)
#
# Pour chaque combinaison (souche, inducteur) et ses parametres (n_hill,
# gain, K, concs) :
#   - pour chaque concentration c de concs : appeler simulate_cells(...)
#     pour obtenir un TABLEAU des etats finaux de N_cells_curve cellules ;
#     conserver la MOYENNE de ce tableau dans une LISTE moyennes (une valeur
#     par concentration) ;
#   - afficher le nuage de points individuels (plt.scatter, avec alpha
#     faible pour la transparence) ET la courbe de la moyenne (plt.plot)
#     superposes sur le meme sous-graphique.
#
# Important : sans le nuage de points individuels, la bimodalite TMG est
# invisible (la courbe moyenne seule ne la montre pas correctement).
#
# Graphique : grille 2x2 (plt.subplots(2, 2, ...)), un sous-graphique par
# combinaison, place a la position donnee par positions[(souche, inducteur)].
#
# Afficher avec print() : comparez aux courbes du Notebook B. Les deux
# panneaux IPTG devraient ressembler aux courbes de Hill de §B2. Que se
# passe-t-il sur les panneaux TMG a concentration intermediaire ?
# Axes : utiliser ax.set_xscale('log') et ax.set_xlim(0.5, None) pour
# mieux visualiser la difference entre courbe hyperbolique et sigmoide
# (sur axe log-x, la transition Hill n=1 s'etale sur ~2 decennies,
#  la bistabilite Plac x TMG apparait quasi-verticale).


> **Si vous avez des données de plaque reader pour le TMG** (mêmes puits que
> l'IPTG, colonnes dédiées du layout — voir Notebook B), vous pouvez les
> superposer aux courbes simulées en réutilisant `parse_biotek` et `layout`
> du Notebook B. Sinon, la comparaison entre la simulation TMG ci-dessus et
> les données réelles IPTG (Notebook B, §B3) suffit pour discuter la
> différence de comportement entre les deux inducteurs.


### Un interrupteur génétique

Les courbes que vous venez de construire illustrent deux façons très différentes de
répondre à un signal chimique.

Pour l'IPTG — quel que soit le promoteur — et pour lacUV5 avec le TMG, la réponse
est **graduée** : plus la concentration d'inducteur augmente, plus l'expression
augmente progressivement. Chaque cellule suit fidèlement la concentration de son
environnement, comme un dimmer qui suit le bouton.

Pour Plac avec le TMG, c'est différent. La rétroaction positive via LacY crée deux
états stables : une cellule est soit fermement ON, soit fermement OFF. À concentration
intermédiaire de TMG, les deux états coexistent dans la même population — d'où le
nuage de points scindé en deux. Ce n'est plus un dimmer : c'est un **interrupteur**.

Ce type de système s'appelle un **interrupteur génétique** (*genetic switch*). Une
fois basculée dans l'état ON, une cellule peut y rester même si la concentration
d'inducteur redescend légèrement — c'est la bistabilité que vous avez observée en
§C1. Ce mécanisme n'est pas une curiosité de laboratoire : la nature l'utilise partout
où une cellule doit prendre une décision durable et irréversible — différenciation
cellulaire, réponse immunitaire, entrée en sporulation chez les bactéries. Dans tous
ces cas, ce qu'on observe au niveau moléculaire ressemble à ce que vous venez de
simuler.


---
## Section C3 [FACULTATIF — bonus] — Analyse des données de microscopie (ImageJ)

Si la session de microscopie a eu lieu, cette section permet de vérifier
directement, cellule par cellule, les prédictions du §C1 et du §C2 :
observe-t-on vraiment deux populations séparées à [TMG] intermédiaire ?

**Export depuis ImageJ** :
1. Ouvrir l'image, segmenter les cellules (*Analyze Particles*)
2. *Measure* → exporter le tableau en CSV (une ligne par cellule)
3. Répéter pour chaque concentration de TMG

La colonne utile dans le CSV ImageJ est `Mean` (fluorescence moyenne par cellule).

> Cette section est entièrement facultative : si le temps manque, §C1 et §C2
> suffisent à répondre à la question scientifique du notebook.


In [ ]:
# Objectif : charger (ou simuler) les donnees de fluorescence single-cell
# obtenues par analyse ImageJ pour plusieurs concentrations de TMG, et
# tracer les histogrammes de distribution.
#
# Donnees a charger (reelles) :
#   concs_tmg  — LISTE des concentrations de TMG utilisees en microscopie
#                (ex. [0, 5, 20, 100])
#   donnees_ij — DICTIONNAIRE {concentration: DataFrame}, un DataFrame par
#                concentration, charge depuis un fichier CSV exporte d'ImageJ
#                (pd.read_csv). La colonne 'Mean' contient la fluorescence
#                moyenne de chaque cellule detectee.
#
# Si les donnees reelles ne sont pas encore disponibles, utiliser des
# donnees synthetiques de structure equivalente : pour chaque concentration,
# un TABLEAU numpy de valeurs de fluorescence (une valeur par cellule).
#
# Graphique : grille de sous-graphiques (plt.subplots(2, 2, ...)), un
# histogramme par concentration de concs_tmg (plt.hist).
#
# Afficher avec print() vos reponses : a quelle concentration de TMG voit-on
# deux pics separes (distribution bimodale) ? La fraction de cellules "ON"
# augmente-t-elle avec [TMG] ? Ces resultats sont-ils coherents avec les
# predictions du §C1 et du §C2 ?


---
## Pour aller plus loin

1. **Connexion avec Elowitz 2002** : l'analyse single-cell de §C3 est proche de
   l'expérience fondatrice sur le bruit d'expression. Dans ce papier, deux gènes
   rapporteurs identiques dans la même cellule montraient des niveaux différents
   — preuve directe du bruit *intrinsèque*. Notez la différence avec la
   bistabilité de §C1 : ici la divergence vient d'une bifurcation déterministe,
   pas du hasard moléculaire.

2. **Quantifier la bimodalité** : à partir des nuages de points du §C2 (ou des
   histogrammes ImageJ du §C3), pouvez-vous mesurer la fraction de cellules
   "ON" pour chaque concentration de TMG, et tracer cette fraction en fonction
   de [TMG] ? C'est une autre façon de représenter le même phénomène que la
   courbe moyenne — laquelle est la plus informative ?

3. **Limite du modèle** : le §C1 utilise le même K pour décrire l'effet de
   l'inducteur sur le promoteur, qu'il y ait rétro-action ou non. Qu'est-ce qui
   pourrait changer entre un promoteur stimulé directement par l'IPTG et un
   promoteur stimulé indirectement via le TMG intracellulaire amplifié par
   LacY ? Est-ce biologiquement réaliste de garder le même K ?
